# Camera Discovery Harvest URLs Test Notebook

This Google Colab notebook tests the `camera-discovery harvest-urls` CLI workflow. Harvest mode is extraction-only: it bypasses target resolution, geocoding, validation, trust policy, scope enforcement, LLM review, GeoJSON, map output, `cameras.md`, and review ZIP generation.

The notebook is a CLI harness only. It does not implement harvest source logic and does not patch application source files at runtime.


In [ ]:
# Colab setup: run from the checked-out repository root.
from pathlib import Path
import os

REPO_ROOT = Path.cwd()
print('Repository root:', REPO_ROOT)
print('pyproject.toml exists:', (REPO_ROOT / 'pyproject.toml').exists())


In [ ]:
# Install the package in editable mode.
%pip install -e .


In [ ]:
# Verify CLI registration.
!camera-discovery --help
!camera-discovery harvest-urls --help


## Harvest all supported media types

This command collects raw direct media/camera URLs across supported media categories and writes plain URL and metadata outputs. It does not validate streams or write inventory artifacts.


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california \
  --max-urls 10000 \
  --discovery-mode both \
  --enable-browser-capture


## Harvest only HLS / `.m3u8` URLs


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california-hls \
  --max-urls 10000 \
  --media .m3u8


## Harvest a media mix: HLS, images, and generic stream URLs


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california-media-mix \
  --max-urls 10000 \
  --media hls,image,stream


## Inspect output paths and summary counts


In [ ]:
from pathlib import Path
import json

HARVEST_DIR = Path('runs/harvest-california')
paths = {
    'camera_urls.txt': HARVEST_DIR / 'camera_urls.txt',
    'camera_urls.csv': HARVEST_DIR / 'camera_urls.csv',
    'camera_urls.jsonl': HARVEST_DIR / 'camera_urls.jsonl',
    'harvest_summary.json': HARVEST_DIR / 'harvest_summary.json',
    'source_rows.jsonl': HARVEST_DIR / 'source_rows.jsonl',
}
for label, path in paths.items():
    print(f'{label}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}')

summary_path = paths['harvest_summary.json']
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('
Summary counts:')
    for key in ['media_filter', 'raw_records', 'unique_urls', 'pre_filter_unique_urls', 'media_filtered_urls', 'written_urls', 'max_urls', 'unlimited']:
        print(f'{key}:', summary.get(key))
    print('by_media_type:', summary.get('by_media_type', {}))
    print('by_source_host:', summary.get('by_source_host', {}))


## Print the first harvested URLs


In [ ]:
N = 25
url_path = Path('runs/harvest-california/camera_urls.txt')
if url_path.exists():
    for idx, line in enumerate(url_path.read_text().splitlines()[:N], start=1):
        print(f'{idx:03d}: {line}')
else:
    print('No camera_urls.txt file found yet.')


## Optional: zip and download harvest outputs


In [ ]:
from pathlib import Path
import shutil

out_dir = Path('runs/harvest-california')
if out_dir.exists():
    archive = shutil.make_archive(str(out_dir), 'zip', root_dir=out_dir)
    print('Created:', archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print('Download helper unavailable outside Colab:', exc)
else:
    print('Run harvest first; output directory does not exist:', out_dir)
